# SPSF_03 — BiAgram stages 1–4

One model (Qwen2.5-3B-Instruct), one test set with three parts (English, Spanish, code-switched), four stages:

| Stage | Question | Model state | Test items | Status |
|---|---|---|---|---|
| **1. Healthy baseline** | Does the undamaged model know the grammar in both languages? | healthy | EN + ES + code-switched | ready |
| **2. Reproduce Roll et al.** | Does my lesion code reproduce a published result? | Roll-style lesions | English BLiMP | ready |
| **3. Baseline method on my set** | What does *generic* damage do to English vs. Spanish errors? | Roll-style lesions, swept | EN + ES + code-switched | ready |
| **4. My method** | What does damage to *language-specific* vs. *shared* neurons do? | language-specific neuron lesions | EN + ES + code-switched | **draft** |

Stages 1 and 2 must pass before reading anything into 3 and 4.

**Before you run:** Runtime → Change runtime type → **T4 GPU**. Put `biagram_testset_v2.csv` in the Files panel (or in Drive, see Settings). Run the cells in order. Stage 3's full sweep is long, so it saves as it goes and picks up where it left off if Colab disconnects.

**Rule:** every lesion is chosen by a fixed rule *before* looking at the results. Nothing here searches for the lesion that "looks most like aphasia".

In [2]:
!pip -q install datasets

## 0. Settings

In [3]:
MODEL_NAME = "Qwen/Qwen3-4B-Base"   # fallback: "Qwen/Qwen2.5-1.5B"
DTYPE = "float16"                          # use "float32" if any score prints as nan
TESTSET_PATH = "/content/biagram_testset_v2.csv"
USE_DRIVE = True                           # save results to Drive so a disconnect doesn't lose them
DRIVE_DIR = "/content/drive/MyDrive/BiAgram/results"
SEED = 0
METRIC = "norm"                            # accuracy score: "norm" = sum / tokens**0.5 (primary); checks: "sum", "mean"
BATCH_SIZE = 8                            # use 8 if DTYPE = "float32"

In [4]:
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS_DIR = DRIVE_DIR
else:
    RESULTS_DIR = "/content/results"
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Results folder:", RESULTS_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Results folder: /content/drive/MyDrive/BiAgram/results


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=getattr(torch, DTYPE), device_map=DEVICE, low_cpu_mem_usage=True).eval()
N_LAYERS = len(model.model.layers)
print(MODEL_NAME, "on", DEVICE, "|", N_LAYERS, "layers |",
      model.model.layers[0].mlp.gate_proj.out_features, "MLP neurons per layer")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.68k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Qwen/Qwen3-4B-Base on cuda | 36 layers | 9728 MLP neurons per layer


## Shared tools

**Scoring.** Each sentence's log-probability is summed over its tokens, then divided by √(number of tokens). A plain sum favors shorter sentences (omission foils) and a plain mean over-corrects; this middle setting was the least biased in Liu et al.'s comparison. Sum and mean versions are kept too, as robustness checks (`METRIC` in Settings). A start-of-text token goes first so the first word is scored. Plain text, never the chat template.

**Two kinds of lesion:**

- `lesion(layer, component, severity, seed)`: Roll et al.'s method. Zero a random fraction of one weight matrix. Components: `q k v o` (attention), `gate up down` (MLP).
- `lesion_neurons({layer: [neuron indices]})`: silence chosen MLP neurons (zero their output weights). Used in stage 4.

Both undo themselves when the `with` block ends, even after an error.

In [11]:
import os, math, contextlib
import torch
import numpy as np
import pandas as pd

START_TOKEN = globals().get("START_TOKEN", "<|endoftext|>")   # Qwen's document separator, same for base and Instruct

def encode(sentences):
    return [tok(START_TOKEN + s, add_special_tokens=False).input_ids for s in sentences]

def pad_batch(batch):
    pad_id = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    width = max(len(x) for x in batch)
    ids = torch.full((len(batch), width), pad_id, dtype=torch.long)
    mask = torch.zeros((len(batch), width), dtype=torch.long)
    for j, x in enumerate(batch):
        ids[j, :len(x)] = torch.tensor(x)
        mask[j, :len(x)] = 1
    return ids.to(DEVICE), mask.to(DEVICE)

@torch.no_grad()
def score_sentences(sentences, batch_size=None, metric=None):
    # Returns one score per sentence. metric: "sum", "mean", "norm" (= sum / tokens**0.5), or "raw" = (sum, tokens).
    batch_size = batch_size or globals().get("BATCH_SIZE", 32)
    ids_list, out = encode(sentences), []
    for i in range(0, len(ids_list), batch_size):
        ids, mask = pad_batch(ids_list[i:i + batch_size])
        logits = model(input_ids=ids, attention_mask=mask).logits.float()
        logp = torch.log_softmax(logits[:, :-1], dim=-1)
        token_lp = logp.gather(2, ids[:, 1:].unsqueeze(-1)).squeeze(-1)
        real = mask[:, 1:].float()
        out.extend(zip((token_lp * real).sum(1).tolist(), real.sum(1).tolist()))
    if any(math.isnan(t) or math.isinf(t) for t, n in out):
        print("WARNING: nan/inf scores. Set DTYPE = 'float32' and BATCH_SIZE = 8 in Settings and rerun.")
    metric = metric or globals().get("METRIC", "norm")
    if metric == "raw":
        return out
    return [to_metric(t, n, metric) for t, n in out]

def to_metric(total, n_tokens, metric):
    return {"sum": total, "mean": total / n_tokens, "norm": total / n_tokens ** 0.5}[metric]

COMPONENTS = {"q": "self_attn.q_proj", "k": "self_attn.k_proj", "v": "self_attn.v_proj",
              "o": "self_attn.o_proj", "gate": "mlp.gate_proj", "up": "mlp.up_proj", "down": "mlp.down_proj"}

@contextlib.contextmanager
def lesion(layer, component, severity, seed=0):
    W = model.model.layers[layer].get_submodule(COMPONENTS[component]).weight
    original = W.data.clone()
    try:
        if severity > 0:
            g = torch.Generator(device=W.device).manual_seed(seed)
            W.data.mul_((torch.rand(W.shape, generator=g, device=W.device) >= severity).to(W.dtype))
        yield
    finally:
        W.data.copy_(original)

@contextlib.contextmanager
def lesion_neurons(neurons):
    # neurons: {layer: LongTensor of MLP neuron indices}. Zeroes each neuron's output column in down_proj.
    saved = {}
    try:
        for layer, idx in neurons.items():
            W = model.model.layers[layer].mlp.down_proj.weight
            idx = idx.to(W.device)
            saved[layer] = (idx, W.data[:, idx].clone())
            W.data[:, idx] = 0
        yield
    finally:
        for layer, (idx, cols) in saved.items():
            model.model.layers[layer].mlp.down_proj.weight.data[:, idx] = cols

10/18/26 sanity check

In [12]:
@torch.no_grad()
def token_view(sentence):
    ids = encode([sentence])[0]
    x = torch.tensor([ids], device=DEVICE)
    logp = torch.log_softmax(model(x).logits.float()[0, :-1], -1)
    lp = logp.gather(1, x[0, 1:].unsqueeze(1)).squeeze(1).tolist()
    return [(tok.decode([t]), round(v, 2)) for t, v in zip(ids[1:], lp)]

for seq in ["uno, dos, tres, cuatro, cinco, seis, siete, ocho, nueve, diez",
            "one, two, three, four, five, six, seven, eight, nine, ten"]:
    pieces = token_view(seq)
    half = [lp for _, lp in pieces[len(pieces) // 2:]]
    avg = sum(half) / len(half)
    print(f"{seq[:25]}...  second-half average: {avg:.2f} ->", "OK" if avg > -1 else "PROBLEM")
print()
for s in ["El niño abre la ventana todos los días.", "El niño abren la ventana todos los días.",
          "El niño vendió fruta ayer."]:
    print(token_view(s))

uno, dos, tres, cuatro, c...  second-half average: -0.08 -> OK
one, two, three, four, fi...  second-half average: -0.13 -> OK

[('El', -9.09), (' niño', -8.32), (' abre', -9.02), (' la', -0.94), (' ventana', -3.03), (' todos', -9.54), (' los', -0.02), (' días', -0.16), ('.', -1.7)]
[('El', -9.09), (' niño', -8.32), (' ab', -6.33), ('ren', -6.9), (' la', -1.91), (' ventana', -3.97), (' todos', -8.91), (' los', -0.08), (' días', -0.29), ('.', -2.19)]
[('El', -9.09), (' niño', -8.32), (' vend', -9.21), ('ió', -1.41), (' fr', -6.18), ('uta', -1.28), (' ay', -6.84), ('er', -0.02), ('.', -1.32)]


## Load the test set

One row per **pair**: a correct sentence and one foil. An item (one correct sentence) usually has several foils, each tagged with an `error_category`:

- `omission`: something left out (an article, an auxiliary, a verb ending)
- `substitution`: a wrong form put in (wrong person, wrong gender, wrong tense, a default form)
- `switch`: code-switched part only. The foil switches language at a point bilinguals avoid.

`part` is `en`, `es` or `cs`. English and Spanish items with the same `frame_id` are translations of each other.

In [13]:
pairs = pd.read_csv(TESTSET_PATH)
REQUIRED = ["pair_id", "item_id", "frame_id", "part", "feature", "correct", "foil", "foil_type", "error_category"]
missing = [c for c in REQUIRED if c not in pairs.columns]
assert not missing, f"Test set is missing columns: {missing}"
assert pairs.pair_id.is_unique, "pair_id must be unique"
print(len(pairs), "pairs,", pairs.item_id.nunique(), "items")
print(pairs.groupby(["part", "feature"]).item_id.nunique().rename("items").to_frame().T)

958 pairs, 419 items
part             cs                                        en                  \
feature cs_aux_verb cs_det_noun cs_pronoun_verb_pilot article auxiliary tense   
items            11          12                    12      48        48    48   

part                        es                                 
feature verb_agreement article auxiliary tense verb_agreement  
items               48      48        48    48             48  


In [14]:
ID_COLS = ["pair_id", "item_id", "frame_id", "part", "feature", "foil_type", "error_category"]

def score_pairs(pairs):
    # Scores every unique sentence once. lp_* columns use METRIC; sum/mean/norm versions are all kept.
    sentences = pd.unique(pd.concat([pairs.correct, pairs.foil]))
    raw = dict(zip(sentences, score_sentences(list(sentences), metric="raw")))
    out = pairs[ID_COLS].copy()
    for side, col in (("correct", pairs.correct), ("foil", pairs.foil)):
        out[f"tok_{side}"] = col.map(lambda s: raw[s][1])
        for m in ("sum", "mean", "norm"):
            out[f"{m}_{side}"] = col.map(lambda s: to_metric(*raw[s], m))
    out["token_gap"] = out.tok_correct - out.tok_foil
    return set_metric(out, globals().get("METRIC", "norm"))

def set_metric(scored, metric):
    scored = scored.copy()
    scored["lp_correct"], scored["lp_foil"] = scored[f"{metric}_correct"], scored[f"{metric}_foil"]
    scored["correct_wins"] = scored.lp_correct > scored.lp_foil
    return scored

def accuracy(scored):
    return (scored.groupby(["part", "feature", "error_category"]).correct_wins.mean().mul(100).round(1)
            .unstack("error_category"))

def item_accuracy(scored):
    # An item counts as correct only if the correct sentence beats ALL its foils.
    items = scored.groupby(["item_id", "part", "feature"]).correct_wins.all().reset_index()
    return items.groupby("part").correct_wins.mean().mul(100).round(1)

SHIFT_METRIC = "mean"   # lesion shifts use the per-token mean: damage that lowers every token equally cancels out
OSI_FLOOR = 0.01        # OSI is left blank when both shifts are smaller than this (too small to call)

def shifts(healthy, lesioned):
    # One row per pair: how far the lesion moved the model toward the foil (relative to the correct sentence),
    # and whether a pair the healthy model got right flipped to wrong.
    c, f = f"{SHIFT_METRIC}_correct", f"{SHIFT_METRIC}_foil"
    m = healthy.merge(lesioned[["pair_id", c, f, "correct_wins"]], on="pair_id", suffixes=("_h", "_l"))
    m["shift"] = (m[f + "_l"] - m[c + "_l"]) - (m[f + "_h"] - m[c + "_h"])
    m["flipped"] = m.correct_wins_h & ~m.correct_wins_l
    return m

def osi(healthy, lesioned):
    # Item-level OSI, only for items that carry both an omission and a substitution foil.
    m = shifts(healthy, lesioned)
    m = m[m.error_category.isin(["omission", "substitution"])]
    per = m.pivot_table(index=["item_id", "frame_id", "part", "feature"], columns="error_category",
                        values="shift", aggfunc="mean").reset_index()
    per = per.dropna(subset=["omission", "substitution"])
    denom = per.omission.abs() + per.substitution.abs()
    per["osi"] = ((per.omission - per.substitution) / denom).where(denom > OSI_FLOOR)
    return per

def language_drift(healthy, lesioned):
    # Matched frames: how much the lesion shifts preference from the Spanish sentence to its English translation
    # (per-token mean, so length differences between languages don't count). Positive = drifted toward English.
    # It doesn't detect drift into other languages (e.g. Chinese).
    def correct_lp(s):
        c = s[s.part.isin(["en", "es"])].drop_duplicates("item_id")
        return c.pivot_table(index="frame_id", columns="part", values=f"{SHIFT_METRIC}_correct", aggfunc="mean")
    h, l = correct_lp(healthy), correct_lp(lesioned)
    return float(((l.en - l.es) - (h.en - h.es)).mean())

# Pre-registered, per feature (Test Set Design Rules). "+" = toward omission, "-" = toward substitution.
PREDICTIONS = {("article", "en"): "+", ("article", "es"): "-",
               ("auxiliary", "en"): "+", ("auxiliary", "es"): "+",
               ("verb_agreement", "es"): "-", ("tense", "es"): "-"}
NOT_TESTED = {("verb_agreement", "en"): "confounded",   # singular items carry omission foils, plural items substitution foils
              ("tense", "en"): "exploratory"}

def verdict(pred, lo, hi):
    if pred not in ("+", "-") or np.isnan(lo):
        return ""
    if (pred == "+" and lo > 0) or (pred == "-" and hi < 0):
        return "yes"
    if (pred == "+" and hi < 0) or (pred == "-" and lo > 0):
        return "no"
    return "unclear"

def feature_osi_from_shifts(m, n_boot=1000, seed=0):
    # Per language x feature. Main number: omit_minus_sub = mean shift toward omission foils minus toward
    # substitution foils, with a bootstrap 95% CI. OSI (scaled -1..+1) and flip rates are secondary checks.
    rng, rows = np.random.default_rng(seed), []
    m = m[m.part.isin(["en", "es"])]
    for (part, feat), g in m.groupby(["part", "feature"]):
        cat = {c: g[g.error_category == c] for c in ("omission", "substitution", "nonfinite")}
        row = {"part": part, "feature": feat}
        for c, short in (("omission", "omit"), ("substitution", "sub"), ("nonfinite", "nonfinite")):
            row[f"shift_{short}"] = cat[c]["shift"].mean() if len(cat[c]) else np.nan
            row[f"flip_{short}"] = cat[c].flipped.mean() if len(cat[c]) else np.nan
        diff = lo = hi = osi_val = np.nan
        a, b = cat["omission"]["shift"].to_numpy(), cat["substitution"]["shift"].to_numpy()
        if len(a) and len(b):
            diff = a.mean() - b.mean()
            boots = (a[rng.integers(0, len(a), (n_boot, len(a)))].mean(1)
                     - b[rng.integers(0, len(b), (n_boot, len(b)))].mean(1))
            lo, hi = np.percentile(boots, [2.5, 97.5])
            denom = abs(a.mean()) + abs(b.mean())
            osi_val = diff / denom if denom > OSI_FLOOR else np.nan
        pred = PREDICTIONS.get((feat, part), NOT_TESTED.get((feat, part), "exploratory"))
        row.update(omit_minus_sub=diff, ci_low=lo, ci_high=hi, osi=osi_val, predicted=pred, matches=verdict(pred, lo, hi))
        rows.append(row)
    return pd.DataFrame(rows)

def feature_osi(healthy, lesioned, n_boot=1000):
    return feature_osi_from_shifts(shifts(healthy, lesioned), n_boot=n_boot)

def feature_accuracy(scored):
    items = scored.groupby(["item_id", "part", "feature"]).correct_wins.all().reset_index()
    return items.groupby(["part", "feature"]).correct_wins.mean().mul(100).rename("item_acc").reset_index()

def profile(healthy, lesioned):
    # One "patient profile card": accuracy, error-type shift and prediction check per language x feature.
    card = feature_accuracy(lesioned).merge(feature_osi(healthy, lesioned), on=["part", "feature"], how="left")
    return card.fillna({"predicted": "", "matches": ""})   # code-switched rows: accuracy only

# Stage 1 — Healthy baseline

**Validation gate:** in the English and Spanish parts, the healthy model must prefer the correct sentence over **all** its foils on **≥ 90%** of items. The code-switched part is reported but not gated: its contrasts are preferences, and a bilingual human annotator only reached 85% on a similar set (Sterner & Teufel 2025). A part that fails means the items or the model need fixing before any lesion work. Check the failing items first; they're usually sentence problems, not model problems.

In [15]:
healthy = score_pairs(pairs)
print("Pair accuracy (% correct beats foil), by error type:")
print(accuracy(healthy), "\n")
print("Item accuracy (correct beats ALL foils), by feature:")
print(feature_accuracy(healthy).pivot(index="feature", columns="part", values="item_acc").round(1), "\n")
gate = item_accuracy(healthy)
for part, acc in gate.items():
    status = ("PASS" if acc >= 90 else "FAIL") if part in ("en", "es") else "(reported, not gated)"
    print(f"{part}: {acc:.1f}% of items correct vs. all foils ->", status)
print("\nSame gate under the other scores (a result should hold under all three):")
print(pd.DataFrame({m: item_accuracy(set_metric(healthy, m)) for m in ("norm", "sum", "mean")}))
fails = healthy[~healthy.correct_wins]
print(f"\n{len(fails)} pairs the healthy model gets wrong:")
print(fails.merge(pairs[["pair_id", "correct", "foil"]], on="pair_id")[["pair_id", "correct", "foil"]].to_string(index=False))

Pair accuracy (% correct beats foil), by error type:
error_category              nonfinite  omission  substitution  switch
part feature                                                         
cs   cs_aux_verb                  NaN       NaN           NaN    40.9
     cs_det_noun                  NaN       NaN           NaN    91.7
     cs_pronoun_verb_pilot        NaN       NaN           NaN     0.0
en   article                      NaN     100.0         100.0     NaN
     auxiliary                    NaN     100.0         100.0     NaN
     tense                      100.0      97.9          95.8     NaN
     verb_agreement             100.0      95.8          95.8     NaN
es   article                      NaN     100.0         100.0     NaN
     auxiliary                    NaN     100.0         100.0     NaN
     tense                      100.0     100.0          91.7     NaN
     verb_agreement              95.8     100.0          95.8     NaN 

Item accuracy (correct beats ALL fo

**Dialect check.** The core set uses general Latin American Spanish, so the model should prefer Latin American forms. Both versions are grammatical; this is a preference check.

In [16]:
DIALECT_PAIRS = [("Ustedes tienen razón.", "Vosotros tenéis razón."),
                 ("¿Ustedes quieren comer ahora?", "¿Vosotros queréis comer ahora?"),
                 ("Mi computadora es nueva.", "Mi ordenador es nuevo."),
                 ("Él maneja el carro todos los días.", "Él conduce el coche todos los días.")]
la, sp = score_sentences([a for a, b in DIALECT_PAIRS]), score_sentences([b for a, b in DIALECT_PAIRS])
for (a, b), x, y in zip(DIALECT_PAIRS, la, sp):
    print("Latin American" if x > y else "Spain         ", "preferred |", a, " vs ", b)

Latin American preferred | Ustedes tienen razón.  vs  Vosotros tenéis razón.
Spain          preferred | ¿Ustedes quieren comer ahora?  vs  ¿Vosotros queréis comer ahora?
Spain          preferred | Mi computadora es nueva.  vs  Mi ordenador es nuevo.
Spain          preferred | Él maneja el carro todos los días.  vs  Él conduce el coche todos los días.


**Outside check: MultiBLiMP 1.0.** Public subject–verb agreement pairs built from real sentences (Jumelet et al. 2026): 770 English and 2,541 Spanish. The healthy model should score high on both.

In [17]:
from huggingface_hub import hf_hub_download
def load_multiblimp(lang, n=300):
    path = hf_hub_download(repo_id="jumelet/multiblimp", filename=f"{lang}/data.tsv", repo_type="dataset")
    df = pd.read_csv(path, sep="\t")
    df = df.sample(n=min(n, len(df)), random_state=0)
    return df.rename(columns={"sen": "good", "wrong_sen": "bad"})[["good", "bad"]]
mb = {lang: load_multiblimp(lang) for lang in ("eng", "spa")}
for lang, bp in mb.items():
    good, bad = score_sentences(list(bp.good)), score_sentences(list(bp.bad))
    print(f"MultiBLiMP {lang}: {100 * np.mean([g > b for g, b in zip(good, bad)]):.1f}% correct ({len(bp)} pairs)")

MultiBLiMP eng: 99.7% correct (300 pairs)
MultiBLiMP spa: 96.3% correct (300 pairs)


In [18]:
healthy.to_csv(f"{RESULTS_DIR}/stage1_healthy_qwen3-4b.csv", index=False)

10/18/26 stage 1 error check

In [19]:
@torch.no_grad()
def token_view(sentence):
    ids = encode([sentence])[0]
    x = torch.tensor([ids], device=DEVICE)
    logp = torch.log_softmax(model(x).logits.float()[0, :-1], -1)
    lp = logp.gather(1, x[0, 1:].unsqueeze(1)).squeeze(1).tolist()
    return [(tok.decode([t]), round(v, 2)) for t, v in zip(ids[1:], lp)]

for c, f in [("El niño abre la ventana todos los días.", "El niño abren la ventana todos los días."),
             ("Los niños venden fruta todos los días.", "Los niños vende fruta todos los días."),
             ("El niño vendió fruta ayer.", "El niño vende fruta ayer.")]:
    print("CORRECT", token_view(c))
    print("FOIL   ", token_view(f))
    print()

CORRECT [('El', -9.09), (' niño', -8.32), (' abre', -9.02), (' la', -0.94), (' ventana', -3.03), (' todos', -9.54), (' los', -0.02), (' días', -0.16), ('.', -1.7)]
FOIL    [('El', -9.09), (' niño', -8.32), (' ab', -6.33), ('ren', -6.9), (' la', -1.91), (' ventana', -3.97), (' todos', -8.91), (' los', -0.08), (' días', -0.29), ('.', -2.19)]

CORRECT [('Los', -12.04), (' niños', -6.42), (' vend', -10.34), ('en', -1.72), (' fr', -3.94), ('uta', -0.9), (' todos', -5.57), (' los', -0.01), (' días', -0.52), ('.', -1.19)]
FOIL    [('Los', -12.04), (' niños', -6.42), (' v', -5.68), ('ende', -9.04), (' fr', -5.8), ('uta', -1.33), (' todos', -6.03), (' los', -0.01), (' días', -0.53), ('.', -1.31)]

CORRECT [('El', -9.09), (' niño', -8.32), (' vend', -9.21), ('ió', -1.41), (' fr', -6.18), ('uta', -1.28), (' ay', -6.84), ('er', -0.02), ('.', -1.32)]
FOIL    [('El', -9.09), (' niño', -8.32), (' v', -6.05), ('ende', -3.59), (' fr', -4.63), ('uta', -1.16), (' ay', -9.79), ('er', -0.41), ('.', -3.22)]

In [20]:
# Does the model use the subject's number at all? Uses the scores already in `healthy`; no model run.
a = healthy.merge(pairs[["pair_id", "correct", "foil", "subject_number"]], on="pair_id")
a = a[(a.part == "es") & a.foil_type.isin(["wrong_number", "wrong_aux_number"])].copy()
a["w_c"] = a.correct.str.split().str[2]
a["w_f"] = a.foil.str.split().str[2]
a["verb"] = a.apply(lambda r: "/".join(sorted([r.w_c, r.w_f])), axis=1)
gap = a.sum_correct - a.sum_foil
a["prefers_plural_form"] = gap.where(a.subject_number == "pl", -gap)
t = a.pivot_table(index="verb", columns="subject_number", values="prefers_plural_form", aggfunc="mean").dropna()
t["uses_subject"] = t.pl > t.sg
print(t.round(2).to_string())
print(f"\nVerbs where a plural subject pushes toward the plural form: {t.uses_subject.mean():.0%} ({t.uses_subject.sum()} of {len(t)})")

subject_number        pl    sg  uses_subject
verb                                        
abre/abren          4.03 -7.08          True
arregla/arreglan    3.68 -7.33          True
barre/barren        2.00 -8.28          True
bebe/beben          6.59 -5.21          True
cierra/cierran      1.30 -5.01          True
cocina/cocinan      4.30 -8.61          True
compra/compran      3.75 -7.30          True
contesta/contestan  5.01 -7.44          True
escribe/escriben   -0.85 -7.20          True
está/están          4.36 -5.70          True
lava/lavan          4.99 -4.34          True
limpia/limpian      2.24 -8.14          True
vende/venden        5.30 -8.58          True

Verbs where a plural subject pushes toward the plural form: 100% (13 of 13)


In [21]:
# Is it the model or the sentences? Score ONLY the word that differs, given the words before it.
def first_diff(c, f):
    cw, fw = c.split(), f.split()
    i = next(k for k in range(min(len(cw), len(fw))) if cw[k] != fw[k])
    return " ".join(cw[:i + 1]), " ".join(fw[:i + 1])

sub = pairs[(pairs.part == "es") & pairs.foil_type.isin(["wrong_number", "wrong_aux_number"])].copy()
sub["c_prefix"], sub["f_prefix"] = zip(*[first_diff(c, f) for c, f in zip(sub.correct, sub.foil)])
sub["verb_only_ok"] = np.array(score_sentences(list(sub.c_prefix), metric="sum")) > np.array(score_sentences(list(sub.f_prefix), metric="sum"))
sub["whole_ok"] = healthy.set_index("pair_id").loc[sub.pair_id, "correct_wins"].values
print(sub.groupby(["foil_type", "subject_number"])[["verb_only_ok", "whole_ok"]].mean().mul(100).round(1))
print(f"\nVerb word only: {sub.verb_only_ok.mean():.0%}   Whole sentence: {sub.whole_ok.mean():.0%}")
print(sub.loc[~sub.verb_only_ok, ["c_prefix", "f_prefix"]].head(10).to_string())

                                 verb_only_ok  whole_ok
foil_type        subject_number                        
wrong_aux_number pl                     100.0     100.0
                 sg                     100.0     100.0
wrong_number     pl                      91.7      91.7
                 sg                     100.0     100.0

Verb word only: 98%   Whole sentence: 98%
                   c_prefix               f_prefix
177  Los agricultores beben  Los agricultores bebe
187      Las doctoras beben      Las doctoras bebe


# Stage 2 — Reproduce Roll et al. 2026

Roll et al. lesioned this exact model at layers 7, 18 and 28 (K, Gate, Up; 75% severity). **Their result:** Gate at layer 7 dropped syntax accuracy by about 52 points; K and Up stayed near baseline. Hugging Face counts layers from 0; check in Roll et al.'s appendix whether their layer 7 is the same one (if they count from 1, use 6, 17, 27). Their 70 items may not be public, so this uses English BLiMP pairs. **Pass = the same pattern**, not the same numbers. Partial lesions are random, so each runs with 3 seeds.

In [7]:
from datasets import load_dataset
BLIMP_SETS = ["regular_plural_subject_verb_agreement_1", "irregular_plural_subject_verb_agreement_1",
              "determiner_noun_agreement_1", "anaphor_number_agreement"]
blimp = pd.concat([load_dataset("nyu-mll/blimp", s, split="train").select(range(50)).to_pandas() for s in BLIMP_SETS])
blimp = blimp.rename(columns={"sentence_good": "good", "sentence_bad": "bad"})[["UID", "good", "bad"]]
print(len(blimp), "BLiMP pairs")

README.md:   0%|          | 0.00/58.6k [00:00<?, ?B/s]

regular_plural_subject_verb_agreement_1/(…): reconstructing file:   0%|          |  0.00B / 49.5kB            

regular_plural_subject_verb_agreement_1/(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

irregular_plural_subject_verb_agreement_(…): reconstructing file:   0%|          |  0.00B / 50.7kB            

...

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

anaphor_number_agreement/train-00000-of-(…): reconstructing file:   0%|          |  0.00B / 41.5kB            

anaphor_number_agreement/train-00000-of-(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

200 BLiMP pairs


In [ ]:
def blimp_accuracy(bp):
    good, bad = score_sentences(list(bp.good)), score_sentences(list(bp.bad))
    return 100 * float(np.mean([g > b for g, b in zip(good, bad)]))

def run_reproduction(bp, layers=(7, 18, 28), components=("k", "gate", "up"), severity=0.75, seeds=(0, 1, 2)):
    rows = [{"layer": -1, "component": "healthy", "seed": -1, "accuracy": blimp_accuracy(bp)}]
    for layer in layers:
        for comp in components:
            for seed in seeds:
                with lesion(layer, comp, severity, seed):
                    rows.append({"layer": layer, "component": comp, "seed": seed, "accuracy": blimp_accuracy(bp)})
    return pd.DataFrame(rows)

In [ ]:
repro = run_reproduction(blimp)
base = repro.accuracy.iloc[0]
table = repro[repro.layer >= 0].groupby(["layer", "component"]).accuracy.agg(["mean", "std"]).round(1)
table["drop"] = (base - table["mean"]).round(1)
print(f"Healthy BLiMP accuracy: {base:.1f}%\n"); print(table)
g7, k7, u7 = (table.loc[(7, c), "drop"] for c in ("gate", "k", "up"))
print("\nPattern reproduced:", "YES" if g7 > 2 * max(k7, u7, 1) else "NO",
      f"(layer 7 drop: gate {g7}, k {k7}, up {u7})")
repro.to_csv(f"{RESULTS_DIR}/stage2_roll_reproduction.csv", index=False)

# Stage 3 — The baseline method on your test set

Roll et al.'s lesions, applied to **your** English, Spanish and code-switched items. This is the first time anyone has asked what generic damage does to the *type* of grammar error in two languages, so this stage is already a result on its own.

**Pre-registered predictions (per feature, from the Design Rules):** articles OSI > 0 in English and < 0 in Spanish; auxiliaries OSI > 0 in both (the control); Spanish agreement and tense OSI < 0, with tense losing more accuracy than agreement. English agreement and tense are exploratory. For each lesion and feature the summary gives the shift toward omission minus the shift toward substitution, with a 95% bootstrap confidence interval, and marks the prediction **yes** (interval on the predicted side), **no** (interval on the other side) or **unclear**. Shifts use the per-token average, so damage that lowers every word equally cancels out and can't fake an omission effect.

The sweep saves after every condition and skips conditions already saved, so if Colab disconnects, just run the cell again. Start with `QUICK_PASS = True` (about 20–30 minutes on a T4), then the full sweep later.

In [ ]:
def run_sweep(pairs, healthy, out_path, layers, components, severities, seeds):
    done = set()
    if os.path.exists(out_path):
        prev = pd.read_csv(out_path)
        counts = prev.condition.value_counts()
        done = set(counts[counts == len(pairs)].index)
        if len(done) < len(counts):                        # a disconnect can leave a condition half-saved
            prev[prev.condition.isin(done)].to_csv(out_path, index=False)
            print("Removed", len(counts) - len(done), "half-saved condition(s); they will be rerun.")
    for layer in layers:
        for comp in components:
            for sev in severities:
                for seed in (seeds if sev < 1 else seeds[:1]):      # a 100% lesion has no randomness
                    cond = f"L{layer}_{comp}_s{sev}_seed{seed}"
                    if cond in done:
                        continue
                    with lesion(layer, comp, sev, seed):
                        scored = score_pairs(pairs)
                    scored["condition"], scored["layer"], scored["component"] = cond, layer, comp
                    scored["severity"], scored["seed"] = sev, seed
                    scored.to_csv(out_path, mode="a", header=not os.path.exists(out_path), index=False)
                    print("done", cond)
    return pd.read_csv(out_path)

def summarize_sweep(sweep, healthy, n_boot=500):
    # One row per lesion x language x feature. Seeds are pooled (each seed's pairs count as extra samples).
    cards = []
    for (layer, comp, sev), grp in sweep.groupby(["layer", "component", "severity"]):
        sh = pd.concat([shifts(healthy, s) for _, s in grp.groupby("seed")])
        card = feature_osi_from_shifts(sh, n_boot=n_boot)
        acc = pd.concat([feature_accuracy(s) for _, s in grp.groupby("seed")]).groupby(["part", "feature"]).item_acc.mean().reset_index()
        card = acc.merge(card, on=["part", "feature"], how="left").fillna({"predicted": "", "matches": ""})
        card["layer"], card["component"], card["severity"] = layer, comp, sev
        card["drift_to_en"] = np.mean([language_drift(healthy, s) for _, s in grp.groupby("seed")])
        cards.append(card)
    long = pd.concat(cards)
    tested = long[long.predicted.isin(["+", "-"])]
    head = (tested.groupby(["layer", "component", "severity"]).matches
            .agg(predictions_yes=lambda x: (x == "yes").sum(), predictions_no=lambda x: (x == "no").sum(),
                 predictions_tested="count").reset_index())
    return long, head

In [ ]:
import os
QUICK_PASS = True
if QUICK_PASS:
    LAYERS, COMPS, SEVS, SEEDS = (3, 7, 12, 18, 28), ("q", "k", "v", "o", "gate", "up", "down"), (0.5, 1.0), (0,)
else:
    LAYERS, COMPS, SEVS, SEEDS = (3, 7, 12, 18, 28), ("q", "k", "v", "o", "gate", "up", "down"), (0.25, 0.5, 0.75, 1.0), (0, 1, 2)
SWEEP_PATH = f"{RESULTS_DIR}/stage3_sweep_{'quick' if QUICK_PASS else 'full'}.csv"
sweep = run_sweep(pairs, healthy, SWEEP_PATH, LAYERS, COMPS, SEVS, SEEDS)
long3, head3 = summarize_sweep(sweep, healthy)
long3.to_csv(f"{RESULTS_DIR}/stage3_profiles_{'quick' if QUICK_PASS else 'full'}.csv", index=False)
print("For each lesion: how many of the 6 pre-registered predictions are confirmed (CI on the predicted side),")
print("contradicted (CI on the other side), out of 6 tested. The rest are 'unclear'.")
print(head3.sort_values(["predictions_yes", "predictions_no"], ascending=[False, True]).head(15).to_string(index=False))
print("\nWarning: with many lesions, some will match by chance. Compare against random-neuron lesions in Stage 4 before reading anything into one lesion.")
long3.head(20)

# Stage 4 — Your method: language-specific vs. shared neurons (DRAFT)

**Selection rule (fixed before looking at any test result).** Adapted from LAPE (Tang et al. 2024) for two languages:

1. Run the healthy model on ~2,000 English–Spanish **translation pairs** (OPUS-100). Parallel text means the two languages carry the same content, so neurons that differ are tracking the language, not the topic.
2. For each MLP neuron, measure how often it fires on English tokens (`p_en`) and on Spanish tokens (`p_es`).
3. **English-specific** = largest `p_en − p_es`; **Spanish-specific** = largest `p_es − p_en`; **shared** = high in both (largest `min(p_en, p_es)`). Each set has the same number of neurons.
4. **Random controls**: for each targeted set (including shared), 5 random sets with the same number of neurons per layer, drawn from neurons that fire just as often. A targeted result only counts if it falls outside the range of its 5 random controls.

None of the test-set sentences are used to pick neurons.

**What to compare** (RQ2, Explanation A vs. B):

- **Dissociation index** = own-language accuracy loss − other-language loss. A says EN-specific damage hurts English only and ES-specific hurts Spanish only. B says shared damage hurts both, with omission in English and substitution in Spanish.
- **Drift to English:** if Spanish-specific damage makes the model prefer English translations instead of producing grammar errors, it's switching languages, not agrammatism. Report that as its own finding.

This stage is a draft: the selection rule and set sizes are a first proposal for you to review, not final.

In [ ]:
from datasets import load_dataset
opus = load_dataset("Helsinki-NLP/opus-100", "en-es", split="train").select(range(20000))
test_sentences = set(pairs.correct) | set(pairs.foil)
pairs_text = [(t["en"], t["es"]) for t in opus["translation"]
              if 30 <= len(t["en"]) <= 200 and 30 <= len(t["es"]) <= 200
              and t["en"] not in test_sentences and t["es"] not in test_sentences][:2000]
EN_TEXT, ES_TEXT = [a for a, b in pairs_text], [b for a, b in pairs_text]
print(len(EN_TEXT), "translation pairs for neuron selection")

In [ ]:
@torch.no_grad()
def activation_probs(texts, batch_size=16):
    # Fraction of tokens on which each MLP neuron fires (gate output > 0, so SiLU > 0). Shape [layers, neurons].
    layers = model.model.layers
    counts = torch.zeros(len(layers), layers[0].mlp.gate_proj.out_features, device=DEVICE)
    state = {"n": 0}
    def make_hook(i):
        def hook(module, inp, out):
            counts[i] += (out > 0)[state["mask"]].float().sum(0)
        return hook
    handles = [layer.mlp.gate_proj.register_forward_hook(make_hook(i)) for i, layer in enumerate(layers)]
    try:
        ids_list = [x[:128] for x in encode(texts)]
        for b in range(0, len(ids_list), batch_size):
            ids, pad_mask = pad_batch(ids_list[b:b + batch_size])
            count_mask = pad_mask.bool().clone()
            count_mask[:, 0] = False                 # skip the start-of-text token
            state["mask"] = count_mask
            state["n"] += int(count_mask.sum())
            model(input_ids=ids, attention_mask=pad_mask)
    finally:
        for h in handles:
            h.remove()
    return (counts / state["n"]).cpu()

def to_layer_dict(flat_idx, n_neurons):
    out = {}
    for f in flat_idx.tolist():
        out.setdefault(f // n_neurons, []).append(f % n_neurons)
    return {l: torch.tensor(v, dtype=torch.long) for l, v in out.items()}

def select_neurons(p_en, p_es, frac=0.005, min_prob=0.2, n_random=5, seed=0):
    # Targeted sets are disjoint and the same size. Random controls match each set's per-layer counts and are drawn
    # from neurons that are just as active (fire on >= min_prob of tokens in at least one language).
    L, n = p_en.shape
    k = max(1, int(frac * L * n))
    active = torch.maximum(p_en, p_es) >= min_prob
    used = torch.zeros(L, n, dtype=torch.bool)
    def top(score, valid):
        valid = valid & ~used
        assert int(valid.sum()) >= k, f"only {int(valid.sum())} eligible neurons for a set of {k}; lower frac or min_prob"
        sc = score.clone().float(); sc[~valid] = -float("inf")
        idx = torch.topk(sc.flatten(), k).indices
        used.view(-1)[idx] = True
        return idx
    flat = {"en_specific": top(p_en - p_es, p_en >= min_prob),
            "es_specific": top(p_es - p_en, p_es >= min_prob),
            "shared": top(torch.minimum(p_en, p_es), (p_en >= min_prob) & (p_es >= min_prob))}
    sets = {name: to_layer_dict(idx, n) for name, idx in flat.items()}
    rng = np.random.default_rng(seed)
    for name in ("en_specific", "es_specific", "shared"):
        for r in range(n_random):
            rand = {}
            for l, idx in sets[name].items():
                pool = np.where((active[l] & ~used[l]).numpy())[0]
                if len(pool) < len(idx):
                    pool = np.where((~used[l]).numpy())[0]
                rand[l] = torch.tensor(rng.choice(pool, size=len(idx), replace=False), dtype=torch.long)
            sets[f"random_like_{name}_{r}"] = rand
    return sets

def run_stage4(pairs, healthy, sets):
    rows, scored_all, cards = [], [], []
    for name, neurons in sets.items():
        with lesion_neurons(neurons):
            s = score_pairs(pairs)
        s["condition"] = name
        scored_all.append(s)
        a_h, a_l = item_accuracy(healthy), item_accuracy(s)
        loss = {p: a_h.get(p, np.nan) - a_l.get(p, np.nan) for p in ("en", "es", "cs")}
        card = profile(healthy, s)
        tested = card[card.predicted.isin(["+", "-"])]
        kind = name.rsplit("_", 1)[0] if name.startswith("random") else name
        rows.append({"lesion": name, "kind": kind, "n_neurons": sum(len(v) for v in neurons.values()),
                     "acc_loss_en": loss["en"], "acc_loss_es": loss["es"], "acc_loss_cs": loss["cs"],
                     "dissociation_en_minus_es": loss["en"] - loss["es"],
                     "predictions_yes": int((tested.matches == "yes").sum()), "predictions_no": int((tested.matches == "no").sum()),
                     "predictions_tested": len(tested),
                     "drift_to_en": language_drift(healthy, s)})
        card["lesion"] = name
        cards.append(card)
    return pd.DataFrame(rows).round(3), pd.concat(cards), pd.concat(scored_all)

In [ ]:
p_en, p_es = activation_probs(EN_TEXT), activation_probs(ES_TEXT)
torch.save({"p_en": p_en, "p_es": p_es}, f"{RESULTS_DIR}/stage4_activation_probs.pt")
results4 = []
for frac in (0.001, 0.005, 0.01):                  # 0.1%, 0.5%, 1% of all MLP neurons per set
    sets = select_neurons(p_en, p_es, frac=frac, seed=SEED)
    table, cards, scored = run_stage4(pairs, healthy, sets)
    table.insert(0, "fraction", frac)
    results4.append(table)
    cards.assign(fraction=frac).to_csv(f"{RESULTS_DIR}/stage4_profiles_frac{frac}.csv", index=False)
    scored.assign(fraction=frac).to_csv(f"{RESULTS_DIR}/stage4_scored_frac{frac}.csv", index=False)
results4 = pd.concat(results4)
results4.to_csv(f"{RESULTS_DIR}/stage4_summary.csv", index=False)
# Is a targeted lesion's dissociation outside what random lesions of the same size and layers produce?
for frac, t in results4.groupby("fraction"):
    for name in ("en_specific", "es_specific", "shared"):
        target = t[t.lesion == name].dissociation_en_minus_es.iloc[0]
        null = t[t.kind == f"random_like_{name}"].dissociation_en_minus_es
        print(f"{frac:.3f} {name:12s} dissociation {target:+6.1f}   random range {null.min():+6.1f} to {null.max():+6.1f}",
              "<- outside random" if (target > null.max() or target < null.min()) else "")
results4